# Joins in Pandas (Simple Guide)

In pandas, we combine two DataFrames using **`pd.merge()`** — just like SQL joins.

Types of joins:
1. **Inner Join** — only matching rows
2. **Left Join** — all rows from left + matches from right
3. **Right Join** — all rows from right + matches from left
4. **Outer Join** — all rows from both

In [1]:
import pandas as pd

# Two simple DataFrames
students = pd.DataFrame({
    'student_id': [1, 2, 3, 4],
    'name': ['Anita', 'Bibek', 'Chandra', 'Dipesh']
})

marks = pd.DataFrame({
    'student_id': [1, 2, 3, 5],
    'marks': [85, 90, 78, 88]
})



In [2]:
students

,student_id,name
0,1,Anita
1,2,Bibek
2,3,Chandra
3,4,Dipesh


In [3]:
marks

,student_id,marks
0,1,85
1,2,90
2,3,78
3,5,88


## 1. Inner Join
Keeps only rows where `student_id` exists in **both** tables.

Here: ids 1, 2, 3 (id 4 has no marks, id 5 has no student — both dropped).

In [4]:
inner = pd.merge(students, marks, on='student_id', how='inner')
inner

,student_id,name,marks
0,1,Anita,85
1,2,Bibek,90
2,3,Chandra,78


## 2. Left Join
Keeps **all rows from the left** table (`students`). Missing marks become `NaN`.

In [5]:
left = pd.merge(students, marks, on='student_id', how='left')
left

,student_id,name,marks
0,1,Anita,85.0
1,2,Bibek,90.0
2,3,Chandra,78.0
3,4,Dipesh,NaN


## 3. Right Join
Keeps **all rows from the right** table (`marks`). Missing names become `NaN`.

In [6]:
right = pd.merge(students, marks, on='student_id', how='right')
right

,student_id,name,marks
0,1,Anita,85
1,2,Bibek,90
2,3,Chandra,78
3,5,NaN,88


## 4. Outer Join
Keeps **all rows from both** tables. Missing values become `NaN`.

In [7]:
outer = pd.merge(students, marks, on='student_id', how='outer')
outer

,student_id,name,marks
0,1,Anita,85.0
1,2,Bibek,90.0
2,3,Chandra,78.0
3,4,Dipesh,NaN
4,5,NaN,88.0


## Quick Summary

| Join type | `how=` | Keeps |
|-----------|--------|-------|
| Inner | `'inner'` | Only matching rows |
| Left | `'left'` | All left rows |
| Right | `'right'` | All right rows |
| Outer | `'outer'` | All rows from both |



## 5. `concat()` — stacking instead of matching

`merge()` matches rows on a key, side by side.
`concat()` just **stacks** tables — on top of each other (`axis=0`) or next to each
other (`axis=1`). Use it when the two tables have the **same columns**, e.g. January
sales and February sales.

In [8]:
jan = pd.DataFrame({'student_id': [1, 2], 'name': ['Anita', 'Bibek']})
feb = pd.DataFrame({'student_id': [3, 4], 'name': ['Chandra', 'Dipesh']})

# ignore_index=True renumbers the result 0,1,2,3 instead of 0,1,0,1
pd.concat([jan, feb], ignore_index=True)

,student_id,name
0,1,Anita
1,2,Bibek
2,3,Chandra
3,4,Dipesh


## 6. Things that go wrong (and how to spot them)

In [9]:
# (a) Different key names -> use left_on / right_on
orders = pd.DataFrame({'id': [1, 2], 'item': ['Book', 'Pen']})
people = pd.DataFrame({'student_id': [1, 2], 'name': ['Anita', 'Bibek']})

pd.merge(orders, people, left_on='id', right_on='student_id')

,id,item,student_id,name
0,1,Book,1,Anita
1,2,Pen,2,Bibek


In [10]:
# (b) Same column name on both sides -> pandas adds _x and _y suffixes.
# Pass suffixes=... to give them real names.
a = pd.DataFrame({'id': [1, 2], 'score': [10, 20]})
b = pd.DataFrame({'id': [1, 2], 'score': [30, 40]})

print(pd.merge(a, b, on='id'))
print()
print(pd.merge(a, b, on='id', suffixes=('_test1', '_test2')))

   id  score_x  score_y
0   1       10       30
1   2       20       40

   id  score_test1  score_test2
0   1           10           30
1   2           20           40


In [11]:
# (c) indicator=True shows WHERE each row came from — the fastest way to
# check whether a join silently dropped or duplicated rows.
pd.merge(students, marks, on='student_id', how='outer', indicator=True)

,student_id,name,marks,_merge
0,1,Anita,85.0,both
1,2,Bibek,90.0,both
2,3,Chandra,78.0,both
3,4,Dipesh,NaN,left_only
4,5,NaN,88.0,right_only


In [12]:
# (d) Always check the row count before and after a merge.
# If it grew, the key was not unique on one of the sides.
inner = pd.merge(students, marks, on='student_id', how='inner')
print("students:", len(students), " marks:", len(marks), " inner result:", len(inner))

students: 4  marks: 4  inner result: 3


### Key takeaways

| Goal | Code |
|------|------|
| Match rows on a key | `pd.merge(a, b, on="key", how=...)` |
| Different key names | `left_on="id", right_on="student_id"` |
| Rename clashing columns | `suffixes=("_left", "_right")` |
| See which side a row came from | `indicator=True` |
| Stack rows with the same columns | `pd.concat([a, b], ignore_index=True)` |

**Rule of thumb:** check `len()` before and after every merge. A row count that
changed unexpectedly means the key was not unique.